# wandb-watch-model — worked example 2: Watch every trainable submodule of a model

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-watch-model`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

To automatically watch every layer that has learnable parameters, we walk `model.named_modules()` and call `wandb.watch` on each submodule that owns at least one trainable parameter (checked with `m.parameters(recurse=False)`). This is more robust than hardcoding specific layer names, especially for architectures built programmatically.

## Worked solution

**Step 1 — walk named_modules.**
We iterate `model.named_modules()` which yields `(name, module)` pairs for every submodule recursively. The first entry `(name='', module=model)` is the model root — we skip it to avoid double-watching.

**Step 2 — check for own trainable parameters.**
For each submodule `m`, we call `m.parameters(recurse=False)` to get only parameters owned directly by `m` (not inherited from children). If any of these have `requires_grad=True`, the module is trainable and worth watching.

**Step 3 — call wandb.watch and record.**
We call `wandb.watch(m, log='all', log_freq=log_freq)` for each qualifying module and append its name to a list. We return the sorted list of watched module names for verification.

In [ ]:
import sys
from unittest.mock import MagicMock
import torch.nn as nn
sys.modules.setdefault('wandb', MagicMock())
import wandb

class TwoLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(32, 16)
        self.layer2 = nn.Linear(16, 4)
    def forward(self, x):
        return self.layer2(self.layer1(x))

def watch_all_trainable(model, log_freq):
    watched = []
    for name, m in model.named_modules():
        if name == '':
            continue  # skip root
        own_params = list(m.parameters(recurse=False))
        if own_params and any(p.requires_grad for p in own_params):
            wandb.watch(m, log='all', log_freq=log_freq)
            watched.append(name)
    return sorted(watched)

# Exercise it
wandb.watch.reset_mock()
model = TwoLayer()
watched_names = watch_all_trainable(model, log_freq=100)
print('Watched modules:', watched_names)
print('wandb.watch call count:', wandb.watch.call_count)